# 🌊 Predicting Atmospheres

As a fun little application of `exoatlas`, we provide an example of how to estimate the probability a planet has an atmosphere according to the 3D cosmic shoreline model presented in [Berta-Thompson, Wachiraphan, and Murray (2026, hereafter BTWM26)](https://ui.adsabs.harvard.edu/abs/2025arXiv250702136B/abstract). This page shows how to calculate atmosphere probabilities for planets and provides a few neat little visualizations. 

In [ ]:
from exoatlas import * 

## Calculate Atmosphere Probabilities for `exoatlas` Planets

If you have an `exoatlas` population and simply want to calculate the probability its planets have atmospheres according to the shoreline model, then you can just call the `probability_of_atmosphere()` method attached to any `exoatlas` population. Behind the scenes, it will download [posterior samples from Zenodo](https://zenodo.org/records/15858798) and use them to calculate atmosphere probabilities, marginalizing over the parameter uncertainties in the current version of the shoreline. Here's the simplest version of what that looks like. Whenever we apply it, this shoreline accounts  for the planet bolometric flux, planet escape velocity, *and* stellar luminosity in estimating the probability of atmospheric retention. 

In [ ]:
s = SolarSystem()
s.probability_of_atmosphere()

When we apply this shoreline model to the Solar System, everything but Mercury should have an atmosphere. When we apply it to exoplanets, it estimates different probabilities based on each system's stellar and planetary environment.

In [ ]:
e = TransitingExoplanets()

In [ ]:
e['TRAPPIST-1'].probability_of_atmosphere()

In [ ]:
e['LTT 1445Ab'].probability_of_atmosphere()

In [ ]:
e['LHS 1140b'].probability_of_atmosphere()

## Visualizing the BTWM26 Cosmic Shoreline

`exoatlas` has a few default built-in [visualizing.ipynb](visualizations) to help show the shoreline across relevant parameter spaces. Because the shoreline is intrinsically 3D, these default visualizations show multiple slices of it, either in multiple panels or in an animation.




In [ ]:
from exoatlas.visualizations import * 

# pick the name of the planet to highlight
planet_name = 'LTT1445Ab'

# create a subset population to highlight that one planet
highlight = e[planet_name]

# define a single panel of the visualization
m = ShorelineStandardMap()

# construct a grid of multiple slices
g = SliceGridGallery(m, N=4)

# add the planets to the panels
g.build([s, e, highlight])

# add embellishments to the plot, including probability 
g.refine()

# colorbar for probability
g.add_colorbar()

# save the figure
plt.savefig(f'shoreline-highlighting-{planet_name}.pdf')


In [ ]:
from exoatlas.visualizations import * 

# pick the name of the planet to highlight
planet_name = 'LTT1445Ab'

# create a subset population to highlight that one planet
highlight = e[planet_name]

# define a single panel of the visualization
m = ShorelineStandardMap(order='vfL', )

# construct an animated grid of multiple slices
a = SliceAnimatedGallery(m, N=4, figsize=(4,4), dpi=300)

# make animation that steps through slices
a.animate([s, e, highlight], filename=f'animated-shoreline-highlighting-{planet_name}.gif')

# hide static figure
plt.close()

![animated cosmic shoreline plot, with atmosphere probabilities appearing in sand and water colors](animated-shoreline-highlighting-LTT1445Ab.gif)

## Direct Access to Shoreline Probabilities

If you want more nuanced control over what to do with the cosmic shoreline probabilities, you can create your own `Shoreline` object, which contains the posterior samples and can be used for various calculations. We'll show how to do this, along with some of the background for the math used to calculate this probability.

In [ ]:
from exoatlas.calculations.shoreline import Shoreline 

# automatically download default posterior
shore = Shoreline()

The probability model in BTWM26 defines a critical shoreline flux with 
$$ 
 \log_{10} \left(\frac{f_{\sf shoreline}}{f_\oplus}\right) =  \log_{10} \left(\frac{f_{\sf 0}}{f_\oplus}\right)  + p \cdot  \log_{10} \left(\frac{v_{\sf esc}}{v_{\sf esc, \oplus}}\right) + q \cdot  \log_{10} \left(\frac{L_\star}{L_\odot}\right).
$$ 
and a distance for a particular planet above that shoreline flux of
$$
\Delta =  \log_{10} \left(\frac{f_{\sf }}{f_\oplus}\right) - \log_{10}\left(\frac{f_{\sf shoreline}}{f_\oplus}\right) = \log_{10}\left(\frac{f}{f_{\sf shoreline}}\right)
$$
and finally the probability of a planet having an atmosphere as 
$$
p_{\sf i} = P(A_{\sf i} = 1 | \mathbf{x}_{\sf i}, \boldsymbol{\theta} ) = \frac{1}{1+e^{\Delta_{\sf i}/w}}
$$
where $f_0, p, q, w$ are the model parameters. We can retrieve point estimates of these parameters from the posterior.

In [ ]:
point_parameters = shore.best_parameters()
point_parameters

Let's use those parameters to estimate the probability of  planet receiving Earth-like flux with Earth-like escape velocity, but orbiting a much less luminous star with $L_\star = 10^{-2} L_\odot$.

In [ ]:
point_probability = shore.probability_of_atmosphere(log_f=0, log_v=0, log_L=-2, **point_parameters)
point_probability

We can also use the samples of parameters to calculate the probability for many possible combinations of in the parameter posterior distribution. This would allow us to propagate the uncertainty in the parameters into the probability of a planet having an atmosphere.


In [ ]:
sampled_parameters = shore.sampled_parameters()
sampled_parameters

In [ ]:
samples_of_probability = shore.probability_of_atmosphere(log_f=0, log_v=0, log_L=-2, **sampled_parameters)
samples_of_probability

In [ ]:
plt.hist(samples_of_probability, alpha=0.3, label='with parameter uncertainty')
plt.axvline(point_probability, label='without parameter uncertainty')
plt.legend(frameon=False)
plt.xlabel('Probability of Atmosphere')
plt.yticks([])

The `.calculate_probability_of_atmosphere_from_posterior_samples` function provides a simple wrapper to calculate summary statistics for the atmosphere probability, accounting for the shoreline parameter uncertainties. It provides three numbers: the median probability of an atmosphere, the lower $1\sigma$ confidence interval, and the upper $1\sigma$ confidence interval. By default, these are returned as three numbers; with the `latex=True` keyword flag, they can be returned as a tidy LaTeX-friendly string. 

In [ ]:
shore.calculate_probability_of_atmosphere_from_posterior_samples(log_f=0, log_v=0, log_L=-2)

In [ ]:
shore.calculate_probability_of_atmosphere_from_posterior_samples(log_f=0, log_v=0, log_L=-2, latex=True)

Standard `exoatlas` populations can also provide access to distributions of calculated atmosphere probabilities. Simply provide the `distribution=True` keyword to a population's `.probability_of_atmosphere()` method, as shown below. For Mars, the distribution of probabilities incorporates the uncertainties in the shoreline parameters. For the exoplanet, it includes both the uncertainties in the shoreline parameters *and* the uncertainties propagated from the substantial uncertainties on the intrinsic planet parameters.

In [ ]:
plt.figure(figsize=(8,3))

histkw = dict(bins = np.linspace(0, 1), alpha=0.5)
p = e['LTT1445Ab'].probability_of_atmosphere(distribution=True)
plt.hist(p.distribution[0], label='LTT1445Ab', **histkw)

p = s['Mars'].probability_of_atmosphere(distribution=True)
plt.hist(p.distribution[0], label='Mars', **histkw)

plt.legend(frameon=False)
plt.xlabel('Probability of Atmosphere');

These distributions are used when calculating uncertainties on the derived atmosphere probability. Read more about [uncertainties](uncertainties.ipynb) for subtles about how these uncertainties get propagated.

In [ ]:
# calculate atmosphere probabilities *and* asymmetric uncertainties
x = e.radius()
y = e.probability_of_atmosphere()
lower, upper = e.get_uncertainty_lowerupper('probability_of_atmosphere')

plt.figure(figsize=(8,3))
plt.scatter(x, y, marker='.')
plt.errorbar(x, y, [lower, upper], elinewidth=1, linewidth=0)
plt.xscale('log'); plt.yscale('log');
plt.xlabel('Planet Radius (Earth radii)') 
plt.ylabel('Probability of Atmosphere');


## Further Reading

More details about the calculations and considerations going into these shoreline probabilities can be found in [Berta-Thompson, Wachiraphan, and Murray (2026, hereafter BTWM26)](https://ui.adsabs.harvard.edu/abs/2025arXiv250702136B/abstract). Have fun!
